In [46]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold, cross_val_score, train_test_split, RandomizedSearchCV
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, make_scorer, mean_absolute_percentage_error
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, RobustScaler, OrdinalEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from category_encoders import LeaveOneOutEncoder
from scipy.stats import randint as sp_randint, uniform as sp_uniform

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import StackingRegressor,RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor, AdaBoostRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor

from category_encoders import TargetEncoder
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

In [47]:
df = pd.read_csv(r"C:\Users\Jay Patel\Campusx\ml_projects\PropNavigator\data\fs\feature_selected_properties.csv")
df

,total_area_sqft,property_type,sector,plot_area_missing,bathrooms,society,area_per_bedroom,servant_room,bedrooms,furnishing_type,balcony,facing,luxury_category,age_possession,pooja_room,price_in_cr
0,1800.0,Independent House,sector 84,1,4,SS Linden Floors,450.000000,0,4,Unfurnished,2,north-east,Luxury,New Property,0,2.76
1,2700.0,Independent Builder Floor,sector 26,1,3,"Block E DLF City Phase 1, Gurgaon",900.000000,0,3,Unfurnished,2,not available,Budget,New Property,0,4.25
2,4000.0,Independent Builder Floor,sector 43,1,4,"Sushant Lok Phase 1, Gurgaon",1000.000000,1,4,Furnished,2,east,Budget,Relatively New,1,5.25
3,4500.0,Independent House,sector 109,0,5,International City by SOBHA Phase 2,1125.000000,1,4,Unfurnished,3+,west,Budget,Relatively New,0,9.00
4,2002.0,Flat,sector 106,1,3,godrej meridien,667.333333,1,3,Unfurnished,3,east,Budget,New Property,0,3.11
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5526,2250.0,Independent House,sector 33,0,4,Central Park Flower Valley Fleur Villas,562.500000,1,4,Semi-Furnished,3+,north-west,Luxury,Relatively New,1,7.15
5527,1656.0,Independent Builder Floor,sector 76,1,2,Whiteland Blissville,552.000000,0,3,Unfurnished,2,north-east,Luxury,New Property,0,2.15
5528,1024.0,Independent Builder Floor,sector 33,1,2,Signature Global Park,512.000000,0,2,Unfurnished,3,north-east,Luxury,New Property,0,0.95
5529,2650.0,Independent Builder Floor,sector 57,1,4,Huda Flats,662.500000,1,4,Semi-Furnished,2,east,Luxury,New Property,1,3.50


In [48]:
df.columns

Index(['total_area_sqft', 'property_type', 'sector', 'plot_area_missing',
       'bathrooms', 'society', 'area_per_bedroom', 'servant_room', 'bedrooms',
       'furnishing_type', 'balcony', 'facing', 'luxury_category',
       'age_possession', 'pooja_room', 'price_in_cr'],
      dtype='object')

In [49]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5531 entries, 0 to 5530
Data columns (total 16 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   total_area_sqft    5531 non-null   float64
 1   property_type      5531 non-null   object 
 2   sector             5531 non-null   object 
 3   plot_area_missing  5531 non-null   int64  
 4   bathrooms          5531 non-null   int64  
 5   society            5531 non-null   object 
 6   area_per_bedroom   5531 non-null   float64
 7   servant_room       5531 non-null   int64  
 8   bedrooms           5531 non-null   int64  
 9   furnishing_type    5531 non-null   object 
 10  balcony            5531 non-null   object 
 11  facing             5531 non-null   object 
 12  luxury_category    5531 non-null   object 
 13  age_possession     5531 non-null   object 
 14  pooja_room         5531 non-null   int64  
 15  price_in_cr        5531 non-null   float64
dtypes: float64(3), int64(5),

In [50]:
df.describe()

,total_area_sqft,plot_area_missing,bathrooms,area_per_bedroom,servant_room,bedrooms,pooja_room,price_in_cr
count,5531.000000,5531.000000,5531.000000,5531.000000,5531.000000,5531.000000,5531.000000,5531.000000
mean,2498.252034,0.698065,3.957151,677.816905,0.557404,3.757006,0.370096,5.394760
std,1348.246286,0.459139,1.724981,266.084364,0.496739,1.584133,0.482874,5.326207
min,301.000000,0.000000,1.000000,200.000000,0.000000,1.000000,0.000000,0.190000
25%,1667.000000,0.000000,3.000000,540.000000,0.000000,3.000000,0.000000,2.305000
50%,2160.000000,1.000000,4.000000,645.428571,1.000000,3.000000,0.000000,3.490000
75%,2897.500000,1.000000,4.000000,781.750000,1.000000,4.000000,1.000000,6.500000
max,15000.000000,1.000000,20.000000,3375.000000,1.000000,20.000000,1.000000,80.000000


##### Step 1. Prepare Target

In [51]:
X = df.drop(columns= ['price_in_cr'])
y = df['price_in_cr']

y_log = np.log1p(y)

##### Step 2: Create Stratification Bins

###### We bin on original price (not log).

In [52]:
price_bins = pd.qcut(y, q=5, labels=False)
price_bins.value_counts()

price_in_cr
0    1150
3    1118
2    1098
4    1094
1    1071
Name: count, dtype: int64

##### Step 3: Stratified Split


In [53]:
X_train, X_test, y_train_log, y_test_log = train_test_split(
    X,
    y_log,
    test_size=0.2,
    stratify=price_bins,
    random_state=42
)

In [ ]:
# Convert back to original price for inspection

y_train = np.expm1(y_train_log)
y_test = np.expm1(y_test_log)

print("Train distribution:")
print(pd.qcut(y_train, 5).value_counts(normalize=True))

print("\nTest distribution:")
print(pd.qcut(y_test, 5).value_counts(normalize=True))

Train distribution:
price_in_cr
(0.189, 2.1]    0.207957
(4.11, 7.75]    0.202080
(2.9, 4.11]     0.198463
(7.75, 80.0]    0.197785
(2.1, 2.9]      0.193716
Name: proportion, dtype: float64

Test distribution:
price_in_cr
(0.349, 2.1]      0.207769
(7.742, 34.0]     0.200542
(4.122, 7.742]    0.199639
(2.9, 4.122]      0.198735
(2.1, 2.9]        0.193315
Name: proportion, dtype: float64


#### Step 4 : Define Column Groups 


In [55]:
num_cols = X_train.select_dtypes(include=["number"]).columns.tolist()
cat_cols = X_train.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

print("Numeric columns:", num_cols)
print("Categorical columns:", cat_cols)

Numeric columns: ['total_area_sqft', 'plot_area_missing', 'bathrooms', 'area_per_bedroom', 'servant_room', 'bedrooms', 'pooja_room']
Categorical columns: ['property_type', 'sector', 'society', 'furnishing_type', 'balcony', 'facing', 'luxury_category', 'age_possession']


#### Step 5 – Define Preprocessors

In [56]:
linear_preprocessor  = ColumnTransformer([("num", StandardScaler(), num_cols),
                   ("cat",OneHotEncoder(drop= 'first',handle_unknown='ignore'),cat_cols)])

In [57]:
linear_preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. `

In [58]:
tree_preprocessor = ColumnTransformer([('cat',OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value= -1),cat_cols)], remainder="passthrough")

In [59]:
tree_preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. ``""{fea

### Step 6 : Define Model Groups

In [60]:
model_dict = {

    "LinearRegression": LinearRegression(),
    "SVR": SVR(),
    "DecisionTree": DecisionTreeRegressor(random_state=42),
    "RandomForest": RandomForestRegressor(random_state=42),
    "ExtraTrees": ExtraTreesRegressor(random_state=42),
    "GradientBoost": GradientBoostingRegressor(random_state=42),
    "AdaBoost": AdaBoostRegressor(random_state=42),
    "XGBoost": XGBRegressor(random_state=42)
}

#### Step 7 : Evaluation Function

In [ ]:
def inverse_transform_target(y_log):
    """Inverse of log(1+y)"""
    
    return np.expm1(y_log)

def mean_absolute_percentage_error(y_true, y_pred):
    """
    Calculates Mean Absolute Percentage Error (MAPE).
    Handles division by zero by masking zero true values.
    """
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100


def scorer(model_name, model, preprocessor,X_train, X_test,y_train_log, y_test_log):

    """
    Fits a pipeline, performs cross-validation, and scores the model on 
    training and test sets (on original price scale).
    """
    
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("regressor", model)
    ])

    # Cross-validation on training set

    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    cv_r2_log = cross_val_score(
        pipeline,
        X_train,
        y_train_log,
        cv=kf,
        scoring="r2",
        n_jobs=-1
    ).mean()

    # Fit on full training data

    pipeline.fit(X_train, y_train_log)

    # Training predictions

    train_pred_log = pipeline.predict(X_train)
    y_train_pred = inverse_transform_target(train_pred_log)
    y_train_true = inverse_transform_target(y_train_log)

    train_r2 = r2_score(y_train_true, y_train_pred)
    train_mae = mean_absolute_error(y_train_true, y_train_pred)
    train_mape = mean_absolute_percentage_error(y_train_true, y_train_pred)

    # Test predictions

    test_pred_log = pipeline.predict(X_test)
    y_pred = inverse_transform_target(test_pred_log)
    y_true = inverse_transform_target(y_test_log)

    test_r2 = r2_score(y_true, y_pred)
    test_mae = mean_absolute_error(y_true, y_pred)
    test_rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    test_mape = mean_absolute_percentage_error(y_true, y_pred)

    print(f"\n--- {model_name} ---")
    print(f"CV R2 (log): {cv_r2_log:.4f}")
    print(f"Test R2: {test_r2:.4f}")
    print(f"Test MAPE: {test_mape:.2f}%")

    return {
        "Model": model_name,
        "CV R2 (log)": round(cv_r2_log, 4),
        "Train R2": round(train_r2, 4),
        "Train MAPE": round(train_mape, 2),
        "Test R2": round(test_r2, 4),
        "Test MAE": round(test_mae, 4),
        "Test RMSE": round(test_rmse, 4),
        "Test MAPE": round(test_mape, 2)}

#### Step 8: Run Linear Models

In [63]:
linear_results = []

for name, model in model_dict.items():
    result_1 = scorer(name, model, linear_preprocessor, X_train, X_test,y_train_log, y_test_log)
    linear_results.append(result_1)

results_1_df = pd.DataFrame(linear_results).sort_values("Test MAPE")
results_1_df


--- LinearRegression ---
CV R2 (log): 0.8737
Test R2: 0.8251
Test MAPE: 17.65%

--- SVR ---
CV R2 (log): 0.9008
Test R2: 0.8853
Test MAPE: 15.51%

--- DecisionTree ---
CV R2 (log): 0.8147
Test R2: 0.7277
Test MAPE: 20.07%

--- RandomForest ---
CV R2 (log): 0.8859
Test R2: 0.8778
Test MAPE: 15.28%

--- ExtraTrees ---
CV R2 (log): 0.8944
Test R2: 0.8795
Test MAPE: 14.46%

--- GradientBoost ---
CV R2 (log): 0.8683
Test R2: 0.8494
Test MAPE: 22.14%

--- AdaBoost ---
CV R2 (log): 0.7301
Test R2: 0.6887
Test MAPE: 38.84%

--- XGBoost ---
CV R2 (log): 0.8955
Test R2: 0.8802
Test MAPE: 15.63%


,Model,CV R2 (log),Train R2,Train MAPE,Test R2,Test MAE,Test RMSE,Test MAPE
4,ExtraTrees,0.8944,0.9998,0.15,0.8795,0.7746,1.7226,14.46
3,RandomForest,0.8859,0.9671,6.03,0.8778,0.8147,1.7347,15.28
1,SVR,0.9008,0.9512,10.75,0.8853,0.8379,1.6808,15.51
7,XGBoost,0.8955,0.9433,11.34,0.8802,0.8316,1.7176,15.63
0,LinearRegression,0.8737,0.8911,10.51,0.8251,0.9453,2.0754,17.65
2,DecisionTree,0.8147,0.9998,0.15,0.7277,1.0758,2.5899,20.07
5,GradientBoost,0.8683,0.8377,21.49,0.8494,1.0841,1.9259,22.14
6,AdaBoost,0.7301,0.6854,38.59,0.6887,1.7231,2.7693,38.84


#### Step 9 – Run Tree Models

In [64]:
tree_results = []

for name, model in model_dict.items():
    result_2 = scorer(name, model, tree_preprocessor, X_train, X_test,y_train_log, y_test_log)
    tree_results.append(result_2)

results_2_df = pd.DataFrame(tree_results).sort_values("Test MAPE")
results_2_df


--- LinearRegression ---
CV R2 (log): 0.7524
Test R2: 0.6341
Test MAPE: 33.33%

--- SVR ---
CV R2 (log): 0.6789
Test R2: 0.5865
Test MAPE: 34.07%

--- DecisionTree ---
CV R2 (log): 0.8033
Test R2: 0.7314
Test MAPE: 20.07%

--- RandomForest ---
CV R2 (log): 0.8961
Test R2: 0.8607
Test MAPE: 15.89%

--- ExtraTrees ---
CV R2 (log): 0.8831
Test R2: 0.8490
Test MAPE: 16.74%

--- GradientBoost ---
CV R2 (log): 0.8814
Test R2: 0.8233
Test MAPE: 20.17%

--- AdaBoost ---
CV R2 (log): 0.7378
Test R2: 0.6695
Test MAPE: 39.42%

--- XGBoost ---
CV R2 (log): 0.8990
Test R2: 0.8686
Test MAPE: 15.19%


,Model,CV R2 (log),Train R2,Train MAPE,Test R2,Test MAE,Test RMSE,Test MAPE
7,XGBoost,0.8990,0.9886,5.21,0.8686,0.8153,1.7990,15.19
3,RandomForest,0.8961,0.9712,5.85,0.8607,0.8448,1.8526,15.89
4,ExtraTrees,0.8831,0.9998,0.15,0.8490,0.9027,1.9287,16.74
2,DecisionTree,0.8033,0.9998,0.15,0.7314,1.1114,2.5720,20.07
5,GradientBoost,0.8814,0.8758,18.51,0.8233,1.0883,2.0861,20.17
0,LinearRegression,0.7524,0.4350,34.53,0.6341,1.6451,3.0021,33.33
1,SVR,0.6789,0.5943,34.16,0.5865,1.8710,3.1916,34.07
6,AdaBoost,0.7378,0.6935,38.40,0.6695,1.7671,2.8534,39.42


#### Experiment 1: StandardScaler + OrdinalEncoder with linear Preprocessor

In [ ]:
print("=== Experiment 1: StandardScaler + OrdinalEncoder")

preprocessor_1 = ColumnTransformer([
    ("scale", StandardScaler(), num_cols),
    ("ordinal", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), cat_cols),
], remainder="passthrough", verbose_feature_names_out=False)

result_3 = []

for name, model in model_dict.items():
    result_score = scorer(name, model, preprocessor_1, X_train, X_test,y_train_log, y_test_log)
    result_3.append(result_score)

results_3_df = pd.DataFrame(result_3).sort_values("Test MAPE")
results_3_df


=== Experiment 1: StandardScaler + OrdinalEncoder - Linear Models ===

--- LinearRegression ---
CV R2 (log): 0.7524
Test R2: 0.6341
Test MAPE: 33.33%

--- SVR ---
CV R2 (log): 0.2210
Test R2: 0.0747
Test MAPE: 59.85%

--- DecisionTree ---
CV R2 (log): 0.8071
Test R2: 0.7374
Test MAPE: 20.14%

--- RandomForest ---
CV R2 (log): 0.8961
Test R2: 0.8594
Test MAPE: 15.89%

--- ExtraTrees ---
CV R2 (log): 0.8824
Test R2: 0.8481
Test MAPE: 16.92%

--- GradientBoost ---
CV R2 (log): 0.8812
Test R2: 0.8233
Test MAPE: 20.17%

--- AdaBoost ---
CV R2 (log): 0.7368
Test R2: 0.6689
Test MAPE: 39.34%

--- XGBoost ---
CV R2 (log): 0.8998
Test R2: 0.8764
Test MAPE: 15.17%


,Model,CV R2 (log),Train R2,Train MAPE,Test R2,Test MAE,Test RMSE,Test MAPE
7,XGBoost,0.8998,0.9886,5.21,0.8764,0.7975,1.7451,15.17
3,RandomForest,0.8961,0.9715,5.88,0.8594,0.8448,1.8611,15.89
4,ExtraTrees,0.8824,0.9998,0.15,0.8481,0.9106,1.9344,16.92
2,DecisionTree,0.8071,0.9998,0.15,0.7374,1.0968,2.5435,20.14
5,GradientBoost,0.8812,0.8758,18.51,0.8233,1.0886,2.0864,20.17
0,LinearRegression,0.7524,0.4350,34.53,0.6341,1.6451,3.0021,33.33
6,AdaBoost,0.7368,0.6933,38.38,0.6689,1.7621,2.8558,39.34
1,SVR,0.2210,0.0919,61.43,0.0747,2.7605,4.7741,59.85


#### EXPERIMENT 2: TargetEncoder + Standard + Ordinal

In [ ]:
print("=== Experiment 2: Target Encoder + Standard + Ordinal ===")

target_encoding_cols = ["sector","society"]

preprocessor_2 = Pipeline([
    ("target_enc", TargetEncoder(cols=target_encoding_cols)),

    ("ct", ColumnTransformer([
        ("scale", StandardScaler(), num_cols),
        ("ordinal", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), cat_cols),
    ], remainder="passthrough", verbose_feature_names_out=False))
])

results_4 = []

for name, model in model_dict.items():
    res = scorer(name, model, preprocessor_2, X_train, X_test,y_train_log, y_test_log)
    results_4.append(res)

results_4_df = pd.DataFrame(results_4).sort_values("Test MAPE")
results_4_df

=== Experiment 2: Target Encoder + Standard + Ordinal ===

--- LinearRegression ---
CV R2 (log): 0.7997
Test R2: 0.7094
Test MAPE: 28.35%

--- SVR ---
CV R2 (log): 0.5772
Test R2: 0.4885
Test MAPE: 40.56%

--- DecisionTree ---
CV R2 (log): 0.7751
Test R2: 0.6801
Test MAPE: 18.99%

--- RandomForest ---
CV R2 (log): 0.8383
Test R2: 0.8187
Test MAPE: 14.77%

--- ExtraTrees ---
CV R2 (log): 0.8406
Test R2: 0.8060
Test MAPE: 15.10%

--- GradientBoost ---
CV R2 (log): 0.8420
Test R2: 0.8018
Test MAPE: 19.42%

--- AdaBoost ---
CV R2 (log): 0.7719
Test R2: 0.7085
Test MAPE: 30.25%

--- XGBoost ---
CV R2 (log): 0.8552
Test R2: 0.8350
Test MAPE: 14.84%


,Model,CV R2 (log),Train R2,Train MAPE,Test R2,Test MAE,Test RMSE,Test MAPE
3,RandomForest,0.8383,0.9750,4.38,0.8187,0.8678,2.1134,14.77
7,XGBoost,0.8552,0.9926,4.50,0.8350,0.8637,2.0158,14.84
4,ExtraTrees,0.8406,0.9998,0.15,0.8060,0.8970,2.1862,15.10
2,DecisionTree,0.7751,0.9998,0.15,0.6801,1.1077,2.8069,18.99
5,GradientBoost,0.8420,0.8956,15.89,0.8018,1.0744,2.2094,19.42
0,LinearRegression,0.7997,0.7185,27.06,0.7094,1.4395,2.6752,28.35
6,AdaBoost,0.7719,0.8004,27.11,0.7085,1.5444,2.6795,30.25
1,SVR,0.5772,0.5357,36.78,0.4885,1.9983,3.5496,40.56


#### Experiements - 3 TargetEncoder + Standard + Ordinal + One hot Encoding

In [68]:
print("=== Experiment 3: Target Encoder + Standard + Ordinal + One Hot Encoding===")



ordinal_encoding_cols = ["luxury_category","floornum_category"]

one_hot_encoding_cols = ["property_type","balcony","facing","furnishing_type","age_possession"]

target_encoding_cols = ["sector","society"]

preprocessor_3 = Pipeline([
    ("target_enc", TargetEncoder(cols=target_encoding_cols)),

    ("ct", ColumnTransformer([
        ("scale", StandardScaler(), num_cols),
        ("ordinal", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), cat_cols),
        ("Onehot_encoding",OneHotEncoder(drop='first',sparse_output=False),one_hot_encoding_cols),

    ], remainder="passthrough", verbose_feature_names_out=False))
])

results_5 = []

for name, model in model_dict.items():
    res = scorer(name, model, preprocessor_3, X_train, X_test,y_train_log, y_test_log)
    results_5.append(res)

results_3_df = pd.DataFrame(results_5).sort_values("Test MAPE")
results_3_df


=== Experiment 3: Target Encoder + Standard + Ordinal + One Hot Encoding===

--- LinearRegression ---
CV R2 (log): 0.8041
Test R2: 0.7096
Test MAPE: 27.80%

--- SVR ---
CV R2 (log): 0.5780
Test R2: 0.4901
Test MAPE: 40.33%

--- DecisionTree ---
CV R2 (log): 0.7831
Test R2: 0.7069
Test MAPE: 18.61%

--- RandomForest ---
CV R2 (log): 0.8408
Test R2: 0.8189
Test MAPE: 14.93%

--- ExtraTrees ---
CV R2 (log): 0.8392
Test R2: 0.8165
Test MAPE: 15.35%

--- GradientBoost ---
CV R2 (log): 0.8528
Test R2: 0.8078
Test MAPE: 19.01%

--- AdaBoost ---
CV R2 (log): 0.7737
Test R2: 0.7187
Test MAPE: 29.25%

--- XGBoost ---
CV R2 (log): 0.8636
Test R2: 0.8357
Test MAPE: 14.77%


,Model,CV R2 (log),Train R2,Train MAPE,Test R2,Test MAE,Test RMSE,Test MAPE
7,XGBoost,0.8636,0.9927,4.49,0.8357,0.8661,2.0120,14.77
3,RandomForest,0.8408,0.9744,4.41,0.8189,0.8769,2.1122,14.93
4,ExtraTrees,0.8392,0.9998,0.15,0.8165,0.8911,2.1258,15.35
2,DecisionTree,0.7831,0.9998,0.15,0.7069,1.0692,2.6868,18.61
5,GradientBoost,0.8528,0.8956,15.66,0.8078,1.0571,2.1758,19.01
0,LinearRegression,0.8041,0.7298,26.48,0.7096,1.4414,2.6745,27.80
6,AdaBoost,0.7737,0.8051,26.33,0.7187,1.4864,2.6320,29.25
1,SVR,0.5780,0.5371,36.56,0.4901,1.9912,3.5440,40.33


#### Experiements - 4 TargetEncoder + Robust Scaling + Ordinal + One hot Encoding

In [69]:
print("=== Experiment 4: Target Encoder + Robust Scaler + Ordinal + One Hot Encoding ===")




ordinal_encoding_cols = ["luxury_category","floornum_category"]

one_hot_encoding_cols = ["property_type","balcony","facing","furnishing_type","age_possession"]

target_encoding_cols = ["sector","society"]

preprocessor_4 = Pipeline([
    ("target_enc", TargetEncoder(cols=target_encoding_cols)),

    ("ct", ColumnTransformer([
        ("scale", RobustScaler(), num_cols),
        ("ordinal", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), cat_cols),
        ("Onehot_encoding",OneHotEncoder(drop='first',sparse_output=False),one_hot_encoding_cols),

    ], remainder="passthrough", verbose_feature_names_out=False))
])

results_6 = []

for name, model in model_dict.items():
    res = scorer(name, model, preprocessor_4, X_train, X_test,y_train_log, y_test_log)
    results_6.append(res)

results_6_df = pd.DataFrame(results_6).sort_values("Test MAPE")
results_6_df


=== Experiment 4: Target Encoder + Robust Scaler + Ordinal + One Hot Encoding ===



--- LinearRegression ---
CV R2 (log): 0.8041
Test R2: 0.7096
Test MAPE: 27.80%

--- SVR ---
CV R2 (log): 0.5965
Test R2: 0.5125
Test MAPE: 39.31%

--- DecisionTree ---
CV R2 (log): 0.7817
Test R2: 0.7070
Test MAPE: 18.58%

--- RandomForest ---
CV R2 (log): 0.8408
Test R2: 0.8190
Test MAPE: 14.93%

--- ExtraTrees ---
CV R2 (log): 0.8392
Test R2: 0.8165
Test MAPE: 15.35%

--- GradientBoost ---
CV R2 (log): 0.8528
Test R2: 0.8078
Test MAPE: 19.01%

--- AdaBoost ---
CV R2 (log): 0.7748
Test R2: 0.7226
Test MAPE: 28.98%

--- XGBoost ---
CV R2 (log): 0.8636
Test R2: 0.8357
Test MAPE: 14.77%


,Model,CV R2 (log),Train R2,Train MAPE,Test R2,Test MAE,Test RMSE,Test MAPE
7,XGBoost,0.8636,0.9927,4.49,0.8357,0.8661,2.0120,14.77
3,RandomForest,0.8408,0.9743,4.41,0.8190,0.8764,2.1115,14.93
4,ExtraTrees,0.8392,0.9998,0.15,0.8165,0.8911,2.1258,15.35
2,DecisionTree,0.7817,0.9998,0.15,0.7070,1.0680,2.6864,18.58
5,GradientBoost,0.8528,0.8956,15.66,0.8078,1.0571,2.1758,19.01
0,LinearRegression,0.8041,0.7298,26.48,0.7096,1.4414,2.6745,27.80
6,AdaBoost,0.7748,0.8133,26.21,0.7226,1.4566,2.6138,28.98
1,SVR,0.5965,0.5632,35.76,0.5125,1.9402,3.4653,39.31


#### HyperParameter Tuning 

1. Random Forest

In [33]:
# --- 1. Define Custom MAPE Scorer ---

def mean_absolute_percentage_error(y_true, y_pred):
    """Calculates Mean Absolute Percentage Error (MAPE)."""
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask]))


neg_mape_scorer = make_scorer(
    mean_absolute_percentage_error, 
    greater_is_better=False 
)


# --- 2. Define Tuning Pipeline and Parameter Grid ---

tuning_pipeline_rf = Pipeline([
    ("preprocessor", preprocessor_5), 
    ("regressor", RandomForestRegressor(random_state=42))
])

# REGULARIZED PARAMETER GRID FOR RANDOM FOREST

param_grid_rf_regularized = {
    'regressor__n_estimators': sp_randint(400, 800), 
    'regressor__max_depth': sp_randint(15, 25), 
    'regressor__min_samples_split': [10, 20], 
    'regressor__min_samples_leaf': [4, 8, 12], 
    'regressor__max_features': ['sqrt', 1.0]
}

# --- 3. Split Data ---

X_train, X_test, y_train_log, y_test_log = train_test_split(
    X, y_log, test_size=0.2, random_state=42
)

# --- 4. Run Randomized Search using MAPE Scorer ---

kf = KFold(n_splits=20, shuffle=True, random_state=42)

random_search_rf = RandomizedSearchCV(
    estimator = tuning_pipeline_rf, 
    param_distributions = param_grid_rf_regularized, 
    n_iter = 20, 
    scoring = neg_mape_scorer, 
    cv = kf, 
    verbose = 2, 
    random_state = 42, 
    n_jobs = -1
)

print("Starting Randomized Search (Optimizing for Minimized MAPE)...")
random_search_rf.fit(X_train, y_train_log)


# --- 5. Display Results ---

best_cv_mape = -random_search_rf.best_score_ 

print("\n--- RandomForestRegressor Hyperparameter Tuning Results (MAPE Optimized) ---")
print(f"Best CV MAPE Score: {best_cv_mape * 100:.4f}%") 
print("Best Parameters:")
print(random_search_rf.best_params_)

best_estimator_rf = random_search_rf.best_estimator_

# --- Diagnostic Training Set Metrics ---

train_pred_log = best_estimator_rf.predict(X_train)
train_r2_log = r2_score(y_train_log, train_pred_log)
y_train_pred_cr = np.expm1(train_pred_log)
y_train_true_cr = np.expm1(y_train_log)
train_mae = mean_absolute_error(y_train_true_cr, y_train_pred_cr)
train_mape = mean_absolute_percentage_error(y_train_true_cr, y_train_pred_cr)

print("\n--- Diagnostic Training Set Metrics ---")
print(f"Train R2 Score (Log): {train_r2_log:.4f}")
print(f"Train MAE (Cr): {train_mae:.4f}")
print(f"Train MAPE: {train_mape * 100:.2f}%")
print("--------------------------------------")


# --- Final Tuned Model Performance on Test Set ---

pred_log = best_estimator_rf.predict(X_test)
y_pred = np.expm1(pred_log)
y_true = np.expm1(y_test_log)

final_r2 = r2_score(y_true, y_pred)
final_mae = mean_absolute_error(y_true, y_pred)
final_mape = mean_absolute_percentage_error(y_true, y_pred)

print("\n--- Final Tuned Model Performance on Test Set ---")
print(f"Final Test R2: {final_r2:.4f}")
print(f"Final MAE: {final_mae:.4f}")
print(f"Final MAPE: {final_mape * 100:.2f}%")

Starting Randomized Search (Optimizing for Minimized MAPE)...
Fitting 20 folds for each of 20 candidates, totalling 400 fits

--- RandomForestRegressor Hyperparameter Tuning Results (MAPE Optimized) ---
Best CV MAPE Score: 7.2664%
Best Parameters:
{'regressor__max_depth': 18, 'regressor__max_features': 1.0, 'regressor__min_samples_leaf': 4, 'regressor__min_samples_split': 10, 'regressor__n_estimators': 617}

--- Diagnostic Training Set Metrics ---
Train R2 Score (Log): 0.9600
Train MAE (Cr): 0.6157
Train MAPE: 10.07%
--------------------------------------

--- Final Tuned Model Performance on Test Set ---
Final Test R2: 0.8362
Final MAE: 0.8528
Final MAPE: 15.80%


2. XGBoost

In [22]:

# --- 1. Define Custom MAPE Scorer (Re-affirming its definition) ---
def mean_absolute_percentage_error(y_true, y_pred):
    """Calculates Mean Absolute Percentage Error (MAPE)."""
    # Assuming the inverse transform (np.expm1) is handled outside this function if used on log data
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask]))

# Define the custom scorer for RandomizedSearchCV (must be negative for maximization)
# This will return a score like -0.0802 if the MAPE is 8.02%
neg_mape_scorer = make_scorer(
    mean_absolute_percentage_error, 
    greater_is_better=False 
)


# --- 2. Define Pipeline and XGBoost Parameters ---
tuning_pipeline_xgb = Pipeline([
    ("preprocessor", preprocessor_5),
    ("regressor", XGBRegressor(
        random_state=42, 
        objective='reg:squarederror', 
        tree_method='hist'
    ))
])

param_grid_xgb = {
    'regressor__learning_rate': sp_uniform(loc=0.01, scale=0.04),
    'regressor__n_estimators': sp_randint(800, 1500),
    'regressor__max_depth': sp_randint(3, 5),
    'regressor__reg_alpha': [0.1, 1, 5, 10],
    'regressor__subsample': sp_uniform(loc=0.7, scale=0.1),
    'regressor__colsample_bytree': sp_uniform(loc=0.7, scale=0.1),
}

# --- 3. Split Data (If not already split) ---
X_train, X_test, y_train_log, y_test_log = train_test_split(
    X, y_log, test_size=0.2, random_state=42
)

# --- 4. Run Randomized Search ---
kf = KFold(n_splits=20, shuffle=True, random_state=42)

random_search_xgb = RandomizedSearchCV(
    estimator = tuning_pipeline_xgb, 
    param_distributions = param_grid_xgb, 
    n_iter = 20, 
    scoring = neg_mape_scorer, # Optimizing for Minimized MAPE
    cv = kf, 
    verbose = 2, 
    random_state = 42, 
    n_jobs = -1
)

print("Starting Randomized Search for XGBoost (Optimizing for Minimized MAPE)...")
random_search_xgb.fit(X_train, y_train_log)


# --- 5. Display Results (FIXED OUTPUT LABELING) ---
best_cv_neg_mape = random_search_xgb.best_score_
best_cv_mape = -best_cv_neg_mape # Correctly convert negative score to positive MAPE

print("\n--- XGBoostRegressor Hyperparameter Tuning Results (MAPE Optimized) ---")
# Print the corrected MAPE score
print(f"Best CV MAPE Score: {best_cv_mape * 100:.4f}%") 
print("Best Parameters:")
print(random_search_xgb.best_params_)

best_estimator_xgb = random_search_xgb.best_estimator_

# --- Diagnostic Training Set Metrics ---
train_pred_log = best_estimator_xgb.predict(X_train)
train_r2_log = r2_score(y_train_log, train_pred_log)
y_train_pred_cr = np.expm1(train_pred_log)
y_train_true_cr = np.expm1(y_train_log)
train_mae = mean_absolute_error(y_train_true_cr, y_train_pred_cr)
train_mape = mean_absolute_percentage_error(y_train_true_cr, y_train_pred_cr)

print("\n--- Diagnostic Training Set Metrics ---")
print(f"Train R2 Score (Log): {train_r2_log:.4f}")
print(f"Train MAE (Cr): {train_mae:.4f}")
print(f"Train MAPE: {train_mape * 100:.2f}%")
print("--------------------------------------")


# --- Final Tuned Model Performance on Test Set ---
pred_log = best_estimator_xgb.predict(X_test)
y_pred = np.expm1(pred_log)
y_true = np.expm1(y_test_log)

final_r2 = r2_score(y_true, y_pred)
final_mae = mean_absolute_error(y_true, y_pred)
final_mape = mean_absolute_percentage_error(y_true, y_pred)

print("\n--- Final Tuned Model Performance on Test Set ---")
print(f"Final Test R2: {final_r2:.4f}")
print(f"Final MAE: {final_mae:.4f}")
print(f"Final MAPE: {final_mape * 100:.2f}%")

Starting Randomized Search for XGBoost (Optimizing for Minimized MAPE)...
Fitting 20 folds for each of 20 candidates, totalling 400 fits

--- XGBoostRegressor Hyperparameter Tuning Results (MAPE Optimized) ---
Best CV MAPE Score: 8.0198%
Best Parameters:
{'regressor__colsample_bytree': 0.7031313292455558, 'regressor__learning_rate': 0.04369139098379994, 'regressor__max_depth': 4, 'regressor__n_estimators': 961, 'regressor__reg_alpha': 1, 'regressor__subsample': 0.7894827350427649}

--- Diagnostic Training Set Metrics ---
Train R2 Score (Log): 0.9545
Train MAE (Cr): 0.5884
Train MAPE: 11.89%
--------------------------------------

--- Final Tuned Model Performance on Test Set ---
Final Test R2: 0.8740
Final MAE: 0.6960
Final MAPE: 14.25%


3. ExtraTrees

In [45]:
from sklearn.ensemble import ExtraTreesRegressor
from scipy.stats import randint as sp_randint

# --- 1. Define Tuning Pipeline and NEW Regularized Parameter Grid ---
tuning_pipeline_etr = Pipeline([
    ("preprocessor", preprocessor_5),
    ("regressor", ExtraTreesRegressor(random_state=42))
])

# ⭐ REGULARIZED PARAMETER GRID FOR EXTRATREES (STRONGER REGULARIZATION) ⭐
param_grid_etr_regularized_v2 = {
    'regressor__n_estimators': sp_randint(400, 800),
    'regressor__max_depth': sp_randint(15, 25),
    'regressor__min_samples_split': [15, 25, 40], # Increased regularization
    'regressor__min_samples_leaf': [6, 10, 15],   # Increased regularization
    'regressor__max_features': [1.0, 'sqrt']
}


# --- 2. Run Randomized Search using MAPE Scorer ---

kf = KFold(n_splits=20, shuffle=True, random_state=42)

random_search_etr_v2 = RandomizedSearchCV(
    estimator = tuning_pipeline_etr,
    param_distributions = param_grid_etr_regularized_v2, 
    n_iter = 20,
    scoring = neg_mape_scorer,
    cv = kf,
    verbose = 2,
    random_state = 42,
    n_jobs = -1
)

print("Starting Regularized Search for ExtraTrees V2 (Optimizing for Minimized MAPE)...")
random_search_etr_v2.fit(X_train, y_train_log)


# --- 3. Display Results ---

best_cv_neg_mape = random_search_etr_v2.best_score_
best_cv_mape = -best_cv_neg_mape
best_estimator_etr_v2 = random_search_etr_v2.best_estimator_

print("\n--- ExtraTreesRegressor V2 Hyperparameter Tuning Results (MAPE Optimized) ---")
print(f"Best CV MAPE Score: {best_cv_mape * 100:.4f}%")
print("Best Parameters:")
print(random_search_etr_v2.best_params_)

# --- Diagnostic Training Set Metrics ---

train_pred_log = best_estimator_etr_v2.predict(X_train)
train_r2_log = r2_score(y_train_log, train_pred_log)
y_train_pred_cr = np.expm1(train_pred_log)
y_train_true_cr = np.expm1(y_train_log)
train_mae = mean_absolute_error(y_train_true_cr, y_train_pred_cr)
train_mape = mean_absolute_percentage_error(y_train_true_cr, y_train_pred_cr)

print("\n--- Diagnostic Training Set Metrics ---")
print(f"Train R2 Score (Log): {train_r2_log:.4f}")
print(f"Train MAE (Cr): {train_mae:.4f}")
print(f"Train MAPE: {train_mape * 100:.2f}%")
print("--------------------------------------")


# --- Final Tuned Model Performance on Test Set ---

pred_log = best_estimator_etr_v2.predict(X_test)
y_pred = np.expm1(pred_log)
y_true = np.expm1(y_test_log)

final_r2 = r2_score(y_true, y_pred)
final_mae = mean_absolute_error(y_true, y_pred)
final_mape = mean_absolute_percentage_error(y_true, y_pred)

print("\n--- Final Tuned Model V2 Performance on Test Set ---")
print(f"Final Test R2: {final_r2:.4f}")
print(f"Final MAE: {final_mae:.4f}")
print(f"Final MAPE: {final_mape * 100:.2f}%")

Starting Regularized Search for ExtraTrees V2 (Optimizing for Minimized MAPE)...
Fitting 20 folds for each of 20 candidates, totalling 400 fits

--- ExtraTreesRegressor V2 Hyperparameter Tuning Results (MAPE Optimized) ---
Best CV MAPE Score: 8.3944%
Best Parameters:
{'regressor__max_depth': 22, 'regressor__max_features': 1.0, 'regressor__min_samples_leaf': 6, 'regressor__min_samples_split': 25, 'regressor__n_estimators': 766}

--- Diagnostic Training Set Metrics ---
Train R2 Score (Log): 0.9440
Train MAE (Cr): 0.6381
Train MAPE: 12.46%
--------------------------------------

--- Final Tuned Model V2 Performance on Test Set ---
Final Test R2: 0.8700
Final MAE: 0.7152
Final MAPE: 14.69%


In [38]:
XGB_best_params = random_search_xgb.best_params_
XGB_best_params

{'regressor__colsample_bytree': 0.7031313292455558,
 'regressor__learning_rate': 0.04369139098379994,
 'regressor__max_depth': 4,
 'regressor__n_estimators': 961,
 'regressor__reg_alpha': 1,
 'regressor__subsample': 0.7894827350427649}

In [39]:
rf_best_params = random_search_rf.best_params_
rf_best_params

{'regressor__max_depth': 18,
 'regressor__max_features': 1.0,
 'regressor__min_samples_leaf': 4,
 'regressor__min_samples_split': 10,
 'regressor__n_estimators': 617}

In [46]:
etr_best_params = random_search_etr_v2.best_params_
etr_best_params

{'regressor__max_depth': 22,
 'regressor__max_features': 1.0,
 'regressor__min_samples_leaf': 6,
 'regressor__min_samples_split': 25,
 'regressor__n_estimators': 766}

Stacking Ensemble 

In [47]:

XGB_PARAMS = {
    'colsample_bytree': 0.7031313292455558,
    'learning_rate': 0.04369139098379994,
    'max_depth': 4,
    'n_estimators': 961,
    'reg_alpha': 1,
    'subsample': 0.7894827350427649,
 
}

# --- Corrected Random Forest Parameters ---

RF_PARAMS = {
    'max_depth': 18,
    'max_features': 1.0,
    'min_samples_leaf': 4,
    'min_samples_split': 10,
    'n_estimators': 617,
    'random_state': 42
}

Extratree_params = {'max_depth': 22,
 'max_features': 1.0,
 'min_samples_leaf': 6,
 'min_samples_split': 25,
 'n_estimators': 766
 }


# 1. Define the Tuned Estimator Objects

xgb_best = XGBRegressor(**XGB_PARAMS)
rf_best = RandomForestRegressor(**RF_PARAMS)
etr_best = ExtraTreesRegressor(**Extratree_params)

# 2. Define the Estimators list using the OBJECTS

estimators = [
    ('rf', rf_best), 
    ('xgb', xgb_best), 
    ('etr',etr_best)
]

# --- 3. Define Stacking Pipeline ---

stacking_pipeline = Pipeline([
    ("preprocessor", preprocessor_5),
    ("regressor", StackingRegressor(
        estimators=estimators, 
        final_estimator=Ridge(random_state=42), 
        n_jobs=-1,
        cv=5 
    ))
])

X_train, X_test, y_train_log, y_test_log = train_test_split(X, y_log, test_size=0.2, random_state=42)

# --- 3. Fit and Predict ---

print("\n--- Fitting Stacking Regressor (XGB + RF + Ridge) ---")
stacking_pipeline.fit(X_train, y_train_log)


# --- 4. Evaluate Performance ---

best_estimator_stacked = stacking_pipeline.named_steps['regressor']


# Diagnostic Training Set Metrics

train_pred_log = stacking_pipeline.predict(X_train)
train_r2_log = r2_score(y_train_log, train_pred_log)
y_train_pred_cr = np.expm1(train_pred_log)
y_train_true_cr = np.expm1(y_train_log)
train_mae = mean_absolute_error(y_train_true_cr, y_train_pred_cr)
train_mape = mean_absolute_percentage_error(y_train_true_cr, y_train_pred_cr)

print("\n--- Diagnostic Training Set Metrics (Stacked) ---")
print(f"Train R2 Score (Log): {train_r2_log:.4f}")
print(f"Train MAE (Cr): {train_mae:.4f}")
print(f"Train MAPE: {train_mape * 100:.2f}%")
print("--------------------------------------")


# Final Tuned Model Performance on Test Set

pred_log = stacking_pipeline.predict(X_test)
y_pred = np.expm1(pred_log)
y_true = np.expm1(y_test_log)

final_r2 = r2_score(y_true, y_pred)
final_mae = mean_absolute_error(y_true, y_pred)
final_mape = mean_absolute_percentage_error(y_true, y_pred)

print("\n--- Final Stacking Ensemble Performance on Test Set ---")
print(f"Final Test R2: {final_r2:.4f}")
print(f"Final MAE: {final_mae:.4f}")
print(f"Final MAPE: {final_mape * 100:.2f}%")


--- Fitting Stacking Regressor (XGB + RF + Ridge) ---

--- Diagnostic Training Set Metrics (Stacked) ---
Train R2 Score (Log): 0.9589
Train MAE (Cr): 0.5457
Train MAPE: 10.66%
--------------------------------------

--- Final Stacking Ensemble Performance on Test Set ---
Final Test R2: 0.8797
Final MAE: 0.6565
Final MAPE: 13.30%
